# Stage 9: Final model package
Packages the selected model (ResNet18 full fine-tuning, run R18-C) with its inference script, model card and selection record, and **checks the packaged files end to end**: the inference script is run on all 544 test images and must reproduce the saved test predictions.

Needs: GPU or CPU runtime (a few minutes), `final_results.zip`, the dataset, and `R18-C_best.pt` in `My Drive/brain-tumor-project/models/`.
Output: `final_model_package.zip` (about 45 MB). Send the printed check results back.

In [ ]:
import os, sys, json, zipfile, subprocess, hashlib, shutil
from pathlib import Path
import numpy as np, pandas as pd
PKG = Path('/content/final_model'); PKG.mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile /content/final_model/predict.py
"""Inference with the final model (ResNet18, run R18-C).  RESEARCH USE ONLY, not a medical device.

    python predict.py --ckpt R18-C_best.pt --images scan1.jpg scan2.png [--csv out.csv] [--device cpu]

Output per image: predicted class and the four class probabilities. Probabilities are not calibrated and
confident errors occur (see MODEL_CARD.md)."""
import argparse, sys
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import torchvision.models as tvm
from torchvision import transforms as T
from PIL import Image

CLASS_NAMES = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


class PadToSquare:
    def __call__(self, img):
        w, h = img.size; s = max(w, h)
        if w == h: return img
        canvas = Image.new(img.mode, (s, s), (0,) * len(img.getbands())); canvas.paste(img, ((s - w) // 2, (s - h) // 2)); return canvas


TF = T.Compose([PadToSquare(), T.Resize((224, 224)), T.ToTensor(), T.Normalize(MEAN, STD)])


def load_model(ckpt, device='cpu'):
    ck = torch.load(ckpt, map_location=device, weights_only=False)   # a checkpoint produced by this project; only load files you trust
    m = tvm.resnet18(weights=None); m.fc = nn.Linear(512, 4); m.load_state_dict(ck['model_state']); return m.to(device).eval()


@torch.no_grad()
def predict(model, paths, device='cpu', batch_size=32):
    out = []
    for i in range(0, len(paths), batch_size):
        x = torch.stack([TF(Image.open(p).convert('RGB')) for p in paths[i:i + batch_size]]).to(device)
        out.append(torch.softmax(model(x), 1).cpu().numpy().astype(np.float64))
    return np.concatenate(out)


if __name__ == '__main__':
    ap = argparse.ArgumentParser(); ap.add_argument('--ckpt', required=True); ap.add_argument('--images', nargs='+', required=True)
    ap.add_argument('--csv'); ap.add_argument('--device', default='cpu'); a = ap.parse_args()
    print('RESEARCH PROTOTYPE: not for clinical use.', file=sys.stderr)
    P = predict(load_model(a.ckpt, a.device), a.images, a.device)
    rows = [dict(image=p, prediction=CLASS_NAMES[int(P[i].argmax())], **{f'prob_{c}': float(P[i, k]) for k, c in enumerate(CLASS_NAMES)}) for i, p in enumerate(a.images)]
    for r in rows: print(f"{r['image']}: {r['prediction']}  " + '  '.join(f"{c}={r['prob_' + c]:.3f}" for c in CLASS_NAMES))
    if a.csv:
        import csv
        with open(a.csv, 'w', newline='') as f: w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

In [ ]:
%%writefile /content/final_model/MODEL_CARD.md
# Model card: brain MRI four-class classifier (student research project)

**Not a medical device. Not validated for clinical use. Do not use it to make or support diagnostic or treatment decisions.**

## Model
- ResNet18 (torchvision), ImageNet-pretrained, all layers fine-tuned, final layer replaced by a 4-way linear layer.
- Classes (output order): Glioma, Meningioma, No Tumor, Pituitary.
- Input: a 2D MRI slice (JPEG/PNG). Preprocessing: pad to a centred black square, resize to 224 x 224, scale to [0,1], ImageNet mean/std normalisation. No random operations at inference.
- Training: Adam, learning rate 1e-3 for the new layer and 1e-4 for pretrained layers, batch size 32, ReduceLROnPlateau, early stopping on validation macro-F1, conservative augmentation (small crop, rotation of at most 10 degrees, 5% shift, slight brightness/contrast). No horizontal flip.
- Checkpoint: run R18-C, epoch 14, selected on validation macro-F1 (0.9932).

## Data
Kaggle dataset `ahmedsorour1/mri-for-brain-tumor-with-bounding-boxes`. Exact duplicates were removed, near-duplicates were grouped, and a stratified group-aware split of about 70/15/15 was used. The dataset has no patient identifiers.

## Measured performance (held-out test split, n = 544, one checkpoint)
Accuracy 0.9798, macro-F1 0.9797, one-vs-rest macro ROC-AUC 0.9994. Mean over three seeds of the same recipe: accuracy 0.9767, macro-F1 0.9773. Per-class results are in `final_model_selection.md`.

## Known limitations
- Possible same-patient images across splits and source heterogeneity could not be ruled out, so the accuracy may be optimistic. No external test set was used.
- Lowest sensitivity for Meningioma. Some errors have high predicted probability.
- Grad-CAM heatmaps agree only weakly with the annotated boxes and must not be read as tumour localisation.
- Only four classes, one dataset, 2D slices. Behaviour on other scanners, sequences, planes, ages, or post-operative images is unknown.

In [ ]:
%%writefile /content/final_model/final_model_selection.md
# Stage 9: Final model selection record

## Decision
**Final model: ResNet18, ImageNet-pretrained, full fine-tuning, run R18-C (seed 42, Stage 6 execution), best-validation checkpoint.**
This is a research model for a student project. It is not a medical device and was not tested on any data outside this dataset.

## Candidates and evidence
| Configuration | runs | trainable params (M) | val macro-F1 | test accuracy | test macro-F1 | test ROC-AUC |
|---|---|---|---|---|---|---|
| CNN from scratch | 1 | 0.98 | 0.9358 | 0.9283 | 0.9238 | 0.9902 |
| CNN, weighted loss | 1 | 0.98 | 0.9547 | 0.9301 | 0.9264 | 0.9909 |
| ResNet18, frozen backbone | 1 | 0.0 | 0.8826 | 0.8768 | 0.8846 | 0.9779 |
| ResNet18, layer4 + fc | 1 | 8.4 | 0.9726 | 0.9559 | 0.9569 | 0.9961 |
| ResNet18, full fine-tuning | 3 | 11.18 | 0.9924 ± 0.0007 | 0.9767 ± 0.0028 | 0.9773 ± 0.0022 | 0.9988 ± 0.0006 |
| ResNet50, frozen backbone | 1 | 0.01 | 0.9127 | 0.8879 | 0.8922 | 0.9816 |
| ResNet50, layer4 + fc | 1 | 14.97 | 0.9830 | 0.9761 | 0.9770 | 0.9988 |
| ResNet50, full fine-tuning | 3 | 23.52 | 0.9906 ± 0.0008 | 0.9792 ± 0.0011 | 0.9794 ± 0.0013 | 0.9995 ± 0.0001 |

Values are mean ± standard deviation over runs when there are three. Test numbers are reported for transparency; they were not used for the choice.

## Rule and reasoning
1. Fine-tuning strategy: full fine-tuning gave the highest validation macro-F1 for both ResNets and was chosen over frozen and layer4-only training.
2. Hyperparameters: the rule fixed before Stage 6 was to adopt a change only if its validation macro-F1 exceeded the reference mean (0.9924) by more than 0.0050 (the larger of the seed range and 0.005). No single-change run met it:

| Run | val macro-F1 | difference from reference mean |
|---|---|---|
| T1-lr1e-5 | 0.9771 | -0.0152 |
| T2-lr1e-3 | 0.9827 | -0.0097 |
| T3-bs16 | 0.9750 | -0.0174 |
| T4-adamw | 0.9819 | -0.0105 |
| T5-dropout | 0.9862 | -0.0062 |
| T6-weighted | 0.9853 | -0.0071 |
| T7-hflip | 0.9947 | +0.0023 |

   Six of the seven variants were lower than the reference. The horizontal-flip variant (T7) was higher by less than the threshold, so it was not adopted. The Stage 4 settings were kept.
3. Architecture: validation macro-F1 means are 0.9924 (ResNet18) and 0.9906 (ResNet50); test macro-F1 means are 0.9773 and 0.9794. The differences are of the same size as the seed-to-seed spread, so the data do not separate the two. ResNet18 was chosen because it has about 48% of the parameters at equivalent measured performance. This is a parsimony choice, not a finding that ResNet18 is better.
4. Seed: among the three ResNet18 seeds, R18-C has the highest validation macro-F1 (0.9932), which is the criterion used.

## What to report as expected performance
The single checkpoint R18-C has test accuracy 0.9798 and macro-F1 0.9797. It is also the best of its three seeds on test, so it is likely at the upper end of what this recipe gives. The mean over three seeds (0.9767 accuracy, 0.9773 macro-F1, standard deviation 0.0028 and 0.0022) is the fairer expectation for a newly trained model. Same-seed reruns of this configuration also differed (R18-C-orig: accuracy 0.9706), so differences of about one percentage point should not be read as real.

## Per-class test results of the final checkpoint
| Class | n | precision | sensitivity (recall) | specificity | F1 |
|---|---|---|---|---|---|
| Glioma | 169 | 0.977 | 0.988 (CI 0.958 to 0.997) | 0.989 | 0.982 |
| Meningioma | 155 | 0.987 | 0.955 (CI 0.910 to 0.978) | 0.995 | 0.971 |
| No Tumor | 74 | 0.986 | 0.973 (CI 0.907 to 0.993) | 0.998 | 0.980 |
| Pituitary | 146 | 0.973 | 1.000 (CI 0.974 to 1.000) | 0.990 | 0.987 |

## Known limitations to state with the model
- No patient identifiers: possible same-patient images across splits and source heterogeneity could not be separated (Stage 7). The test accuracy may be optimistic; an external test set is needed.
- Errors are concentrated in Meningioma and in images from a different filename family (a proxy for source), and errors tend to occur on images less similar to any training image.
- High-confidence errors occur, so the predicted probability is not a reliable warning.
- Grad-CAM agreement with the boxes is weak (Stage 8), so the heatmaps must not be presented as tumour localisation.
- Four classes only; other tumour types, other sequences, other scanners and post-operative images were not covered.

## Inputs

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
CKPT = Path('/content/drive/MyDrive/brain-tumor-project/models/R18-C_best.pt')
assert CKPT.exists(), f'Not found: {CKPT}'
RES = Path('/content/final_results')
if not (RES/'metrics'/'splits.csv').exists():
    from google.colab import files
    print('Upload final_results.zip'); up = files.upload()
    with zipfile.ZipFile(list(up)[0]) as f: f.extractall(RES)
    if not (RES/'metrics').exists():
        sub = [p for p in RES.iterdir() if (p/'metrics').exists()]; assert sub; RES = sub[0]

In [ ]:
DATA_SOURCE = 'kaggle'                                         # 'kaggle' or 'drive'
DRIVE_ZIP = '/content/drive/MyDrive/brain_tumor_dataset.zip'   # only used if DATA_SOURCE == 'drive'
SLUG = 'ahmedsorour1/mri-for-brain-tumor-with-bounding-boxes'
RAW = Path('/content/data_raw'); RAW.mkdir(exist_ok=True)
if not any(p.is_file() for p in RAW.rglob('*')):
    if DATA_SOURCE == 'drive':
        assert Path(DRIVE_ZIP).exists(), f'Not found: {DRIVE_ZIP}'
        with zipfile.ZipFile(DRIVE_ZIP) as f: f.extractall(RAW)
    else:
        if not (os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY')):
            try:
                from google.colab import userdata
                os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME'); os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
            except Exception:
                from google.colab import files
                print('Upload kaggle.json'); up = files.upload()
                kd = Path.home()/'.kaggle'; kd.mkdir(exist_ok=True); (kd/'kaggle.json').write_bytes(up[list(up)[0]]); os.chmod(kd/'kaggle.json', 0o600)
        subprocess.run(['pip','-q','install','kaggle'], check=True)
        r = subprocess.run(['kaggle','datasets','download','-d',SLUG,'-p',str(RAW),'--unzip'], capture_output=True, text=True)
        print(r.stdout[-500:], r.stderr[-800:]); r.check_returncode()
def find_root(base):
    for p in [base] + [q for q in base.rglob('*') if q.is_dir()]:
        if {'train','val'} <= {c.name.lower() for c in p.iterdir() if c.is_dir()}: return p
    raise FileNotFoundError('Train/ and Val/ not found')
ROOT = find_root(RAW); print('Dataset root:', ROOT)

## Check the packaged inference script against the saved predictions

In [ ]:
sys.path.insert(0, '/content/final_model')
import torch, predict as P
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
model = P.load_model(CKPT, dev)
sp = pd.read_csv(RES/'metrics'/'splits.csv'); te = sp[sp.split=='test'].reset_index(drop=True)
probs = P.predict(model, [str(ROOT/r) for r in te.rel], dev)
saved = pd.read_csv(RES/'predictions'/'R18-C_test_predictions.csv').set_index('rel').loc[te.rel]
sprob = saved[[f'prob_{c}' for c in P.CLASS_NAMES]].values
pred = probs.argmax(1)
check = dict(n_test_images=int(len(te)), label_agreement=float((pred == saved.y_pred.values).mean()), n_label_disagreements=int((pred != saved.y_pred.values).sum()),
             max_abs_prob_difference=float(np.abs(probs - sprob).max()), accuracy_from_packaged_script=float((pred == te.label_idx.values).mean()), device=dev)
print(json.dumps(check, indent=1))
assert check['label_agreement'] >= 0.99, 'The packaged script does not reproduce the saved predictions'

## Write metadata, checksum and the package

In [ ]:
sha = hashlib.sha256(CKPT.read_bytes()).hexdigest()
shutil.copy(CKPT, PKG/'R18-C_best.pt')
ck = torch.load(CKPT, map_location='cpu', weights_only=False)
meta = dict(model='ResNet18 (torchvision), full fine-tuning', run_id='R18-C', classes=P.CLASS_NAMES, input_size=224, normalisation=dict(mean=P.MEAN, std=P.STD),
            checkpoint_sha256=sha, checkpoint_epoch=int(ck.get('epoch', -1)), validation_macro_f1=float(ck.get('monitor_value', float('nan'))),
            torch=torch.__version__, verification_against_saved_test_predictions=check,
            intended_use='research and teaching only; not a medical device; not validated for clinical use')
json.dump(meta, open(PKG/'final_model_metadata.json', 'w'), indent=1)
(PKG/'SHA256.txt').write_text(f'{sha}  R18-C_best.pt\n')
print('sha256', sha)
shutil.make_archive('/content/final_model_package', 'zip', PKG)
from google.colab import files; files.download('/content/final_model_package.zip')